# **Import Modules and Libaries**

In [1]:
import numpy as np
import pandas as pd
import ast
import kagglehub

# **Download Datasets**

In [2]:
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

credits = pd.read_csv(f"{path}/tmdb_5000_credits.csv")
movies = pd.read_csv(f"{path}/tmdb_5000_movies.csv")

print(credits.shape, movies.shape)  # Should print: (4803, 4) (4803, 20)


Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.
(4803, 4) (4803, 20)


In [3]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [4]:
# merge both datasets
new_df = movies.merge(credits, on='title')

In [5]:
new_df = new_df[['movie_id','title', 'genres','keywords', 'overview', 'cast', 'crew']]

In [6]:
new_df = new_df.dropna()

In [7]:
# checking for null values
new_df.isnull().sum()

,0
movie_id,0
title,0
genres,0
keywords,0
overview,0
cast,0
crew,0


In [8]:
# checking for duplicates values in data
new_df.duplicated().sum()

np.int64(0)

# **Helper Functions**

In [9]:
# Helper function: Convert string to list and extract genres names
def convert_list(string):
  genre_list = []
  for i in ast.literal_eval(string):
    genre_list.append(i["name"])
  return genre_list

In [10]:
# Helper function: Extract the keywords
def convert_keywords(keywords):
  keyword_list = []
  for i in ast.literal_eval(keywords):
    keyword_list.append(i["name"])
  return keyword_list

In [11]:
def convert_cast(cast):
  actor_list = []
  count = 0
  for i in ast.literal_eval(cast):
    if count != 3:
      actor_list.append(i["name"])
      count += 1
    else:
      break
  return actor_list

In [12]:
def convert_crew(crew):
  director_list = []
  for i in ast.literal_eval(crew):
    if i['job'] == 'Director':
      director_list.append(i["name"])
      break
  return director_list

# **Data Pre-processing**

In [13]:
new_df['genres'] = new_df['genres'].apply(convert_list)

In [14]:
new_df['keywords'] =  new_df['keywords'].apply(convert_keywords)

In [15]:
new_df['cast'] = new_df['cast'].apply(convert_cast)

In [16]:
new_df['crew'] =  new_df['crew'].apply(convert_crew)

In [17]:
new_df['overview'] = new_df['overview'].apply(lambda x:x.split())

In [18]:
new_df['genres'] = new_df['genres'].apply(lambda x:[i.replace(" ","") for i in x])
new_df['keywords'] = new_df['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
new_df['cast'] = new_df['cast'].apply(lambda x:[i.replace(" ","") for i in x])
new_df['crew'] = new_df['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [19]:
new_df['tags'] = new_df['overview'] + new_df['genres'] + new_df['keywords'] + new_df['cast'] + new_df['crew']

In [20]:
df = new_df[['movie_id', 'title', 'tags']]

In [21]:
df['tags'] =  df['tags'].apply(lambda x:" ".join(x))

/tmp/ipykernel_473/346541540.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] =  df['tags'].apply(lambda x:" ".join(x))


In [22]:
df['tags'] = df['tags'].apply(lambda x:x.lower())

/tmp/ipykernel_473/841579467.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x:x.lower())


# **Stemming**

Stemming the tags column so we can get rid of the repetative words in vectorization

In [47]:
import nltk

In [48]:
from nltk.stem.porter import PorterStemmer

In [49]:
ps = PorterStemmer()

In [54]:
ps.stem('loving')

'love'

In [55]:
# Helper function
def stemming(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [57]:
df['tags'] = df['tags'].apply(stemming)

/tmp/ipykernel_473/1455447967.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(stemming)


# **Vectorization**
**Text-Vertorization** 🇰
using bags of word technique

In [58]:
from sklearn.feature_extraction.text import CountVectorizer

In [59]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [60]:
vectors = cv.fit_transform(df['tags']).toarray()

In [61]:
vectors.shape

(4806, 5000)

In [62]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

# **Co-sine Similarity**

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

In [65]:
similarity = cosine_similarity(vectors)

In [104]:
def recommend(movie):

    if movie not in df['title'].values:
        print("Movie not found!")
        return

    movie_index = df[df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(df.iloc[i[0]].title)

In [105]:
recommend('John cena')

Movie not found!
